# Sandbox SDK Quickstart

This notebook demonstrates the Python Sandbox SDK on prokube. It claims a sandbox from a WarmPool, runs stateful Python code, executes shell commands, writes and reads files, and cleans up the sandbox.

The examples are intentionally small so you can verify the SDK path before building a larger agent or automation workflow.

## Prerequisites

You need:

- a workspace with the Sandbox module enabled;
- a ready WarmPool, for example `python-pool`;
- `PROKUBE_API_URL`, including the pkui prefix for external access, for example `https://<cluster-domain>/pkui`;
- `PROKUBE_API_KEY` when running from outside the cluster.

Do not paste real API keys into this notebook. Set them as environment variables or use your notebook environment's secret handling.

## 1. Install the SDK

Run this cell if the SDK is not already installed in your notebook environment.

In [ ]:
%pip install -q "git+https://github.com/prokube/prokube-sdk.git@v0.1.3"

## 2. Configure the Client

The SDK reads configuration from environment variables. In a managed Lab, the notebook can usually detect the workspace and use the in-cluster Agent Gateway service. Set `SANDBOX_POOL` if your WarmPool has a different name.

In [ ]:
import os
from pathlib import Path

SERVICE_ACCOUNT_NAMESPACE = Path(
    "/var/run/secrets/kubernetes.io/serviceaccount/namespace"
)

if not os.environ.get("PROKUBE_API_URL") and os.environ.get("KUBERNETES_SERVICE_HOST"):
    os.environ["PROKUBE_API_URL"] = (
        "http://agentgateway-proxy.agentgateway-system.svc.cluster.local"
    )

if not os.environ.get("PROKUBE_WORKSPACE") and SERVICE_ACCOUNT_NAMESPACE.exists():
    os.environ["PROKUBE_WORKSPACE"] = SERVICE_ACCOUNT_NAMESPACE.read_text().strip()

if (
    not os.environ.get("PROKUBE_API_KEY")
    and not os.environ.get("PROKUBE_USER_ID")
    and not os.environ.get("KF_USER")
):
    # Managed single-user workspaces commonly use the namespace as user id.
    os.environ["PROKUBE_USER_ID"] = os.environ.get("PROKUBE_WORKSPACE", "")

sandbox_pool = os.environ.get("SANDBOX_POOL", "python-pool")

required = ["PROKUBE_API_URL", "PROKUBE_WORKSPACE"]
missing = [name for name in required if not os.environ.get(name)]

if missing:
    raise RuntimeError(
        "Missing required environment variables: "
        + ", ".join(missing)
        + ". Set PROKUBE_API_URL and PROKUBE_WORKSPACE before continuing."
    )

print("Workspace:", os.environ["PROKUBE_WORKSPACE"])
print("API URL:", os.environ["PROKUBE_API_URL"])
print("WarmPool:", sandbox_pool)
print("API key configured:", bool(os.environ.get("PROKUBE_API_KEY")))
print("User ID configured:", bool(os.environ.get("PROKUBE_USER_ID") or os.environ.get("KF_USER")))

## 3. Claim a Sandbox

Claiming from a WarmPool should be faster than creating a cold sandbox. Always clean up the sandbox when the task is done.

In [ ]:
from prokube.sandbox import Sandbox

sbx = Sandbox.from_pool(sandbox_pool)
print("Claimed sandbox:", sbx.name)
print("Initial status:", sbx.status)

## 4. Run Stateful Python Code

`run_code()` uses a stateful Python kernel. Imports and variables persist across calls while the sandbox is running.

In [ ]:
sbx.run_code("import statistics")
sbx.run_code("values = [1, 2, 3, 4, 5]")
result = sbx.run_code("print(statistics.mean(values))")
print(result.stdout)

## 5. Run Shell Commands

Use `commands.run()` for command-line tools inside the sandbox.

In [ ]:
command = sbx.commands.run("python --version")
print("exit code:", command.exit_code)
print(command.stdout)
print(command.stderr)

## 6. Work with Files

Files under `/workspace` are a good default for task data that should survive pause/resume.

In [ ]:
csv_data = "name,score\nalice,10\nbob,12\n"
sbx.files.write("/workspace/scores.csv", csv_data)

content = sbx.files.read("/workspace/scores.csv")
print(content.decode() if isinstance(content, bytes) else content)

files = sbx.files.list("/workspace")
for file_info in files:
    print(file_info)

## 7. Clean Up

Delete the sandbox when you are done. For production code, use `try`/`finally` or the SDK context manager so cleanup still runs after errors.

In [ ]:
sbx.kill()
print("Deleted sandbox:", sbx.name)

## Recommended Pattern

For scripts and agents, prefer a context manager so sandbox cleanup is automatic.

In [ ]:
with Sandbox.from_pool(sandbox_pool) as sandbox:
    result = sandbox.run_code("print('hello from a managed sandbox')")
    print(result.stdout)